# Hindlimb Model Edits 
+ Use this notebook to keep track of code used to edit the hindlimb model
+ Place functions in the `src` directory and import them in the blocks that need them 
+ Document your changes with markdown cells before the code block in the style of:

## Step 0: Load the model
This step loads the model and sets up the environment for editing.

In [ ]:
import opensim as osim
from src.muscle_utils import attachments_to_csv

# Load the model
model = osim.Model('rat_hindlimb.osim')
attachments_to_csv(model, 'muscle_attachments.csv')

## Step 1: Convert Thelen muscles to Millard
There is an error in the equilibrate muscles function for Thelen muscles, and Millard muscles are more prevalent.

In [ ]:
from src.muscle_utils import model_thelen_to_millard

millard_model = model_thelen_to_millard(model)
# Save the modified model
millard_model.printToXML('rat_hindlimb_millard.osim')

## Step 2: Analyze muscle lengths and moment arms
Create plots of the muscle lengths and moment arms to find discontinuities or other potential errors.

In [ ]:
from src.musculoskeletal_graph import MusculoskeletalGraph

millard_tsl_model = millard_model.clone()
graph = MusculoskeletalGraph(millard_tsl_model)

# Iterate through the range of motion to compute lengths and moment arms for all muscles
results = graph.muscle_rom_analysis(min_points=1000)

In [ ]:
import plotly.graph_objects as go
import os
# import plotly.io as pio
# pio.renderers.default = "notebook"
figure_dir = 'figs'

# Plot lengths and moment arms to identify problems
for muscle_name, muscle_data in results.items():
    # Plot parallel coordinates of muscle lengths
    fig1 = go.Figure(data=
        go.Parcoords(
            line=dict(color=muscle_data['length'], colorscale='Viridis', showscale=True),
            dimensions=[dict(
                label=col,
                values=muscle_data[col],
            )
            for col in muscle_data.columns if not 'moment_arm' in col]
        )   
    )   
    fig1.update_layout(title=muscle_name + ' Lengths')
    # fig1.write_html(f"{figure_dir}/{muscle_name}_length.html")
    if not os.path.exists(f"{figure_dir}/{muscle_name}"):
        os.makedirs(f"{figure_dir}/{muscle_name}")
    fig1.write_image(f"{figure_dir}/{muscle_name}/{muscle_name}_length.png")

    moment_arm_cols = [col for col in muscle_data.columns if 'moment_arm' in col]
    coord_cols = [col for col in muscle_data.columns if not 'moment_arm' in col and col != 'length']
    for moment_arm in moment_arm_cols:
        # Plot scatter plot of moment arm versus singular coordinate
        for coord in coord_cols:
            fig2 = go.Figure(data=
                go.Scatter(
                    x=muscle_data[coord],
                    y=muscle_data[moment_arm],
                    mode='markers',
                    marker=dict(color=muscle_data[moment_arm], colorscale='Viridis', showscale=True),
                )
            )
            fig2.update_layout(title=f'{muscle_name} {moment_arm} vs {coord}')
            # fig2.write_html(f"{figure_dir}/{muscle_name}_{moment_arm}_vs_{coord}.html")
            fig2.write_image(f"{figure_dir}/{muscle_name}/{muscle_name}_{moment_arm}_vs_{coord}.png")

## Step 3: Rough estimate tendon slack lengths
This is done initially so that the model can be opened in OpenSim Creator for easier muscle editing. It will need to be redone later to account for changes to moment arms. This method is currently using the whole range of motion of each joint, but you could also specify coordinate combinations that you expect to see in the analyzed motion. Using the below method, you can catch most issues, but if you want to just focus on a specific motion, it might be unnecessary.

In [ ]:
from src.muscle_utils import optimize_model_tsl

results = graph.muscle_rom_analysis(min_points=10)
millard_tsl_model: osim.Model = optimize_model_tsl(millard_tsl_model, results)

millard_tsl_model.printToXML('rat_hindlimb_millard_tsl.osim')

## Potentially problematic muscles
### BFa 
![BFa Length](./figs/BFa/BFa_length.png)

### CF
![CF Length](./figs/CF/CF_length.png)

### STp
![STp Length](./figs/STp/STp_length.png)

### TA
![TA Length](./figs/TA/TA_length.png)

### VL
![VL Length](./figs/VL/VL_length.png)

Because of the hard-coded value in [OpenSim's WrapCylinder.cpp code](https://github.com/opensim-org/opensim-core/blob/f9cd558ec3ea99dda206e5e412e62e23cf19bd7e/OpenSim/Simulation/Wrap/WrapCylinder.cpp#L668):

```c++
// Each muscle segment on the surface of the cylinder should be
// 0.002 meters long. This assumes the model is in meters, of course.
int numWrapSegments = (int)(aWrapResult.wrap_path_length / 0.002);
if (numWrapSegments < 1) numWrapSegments = 1;
```
we need to modify the wrapping surfaces to prevent discontinuities in muscle lengths that appear in the range of motion of the rats during normal gait

We plan to use the attachment points presented in Young's work which build upon Johnson and Greene's original anatomical text. However, the coordinate systems are defined differently and the scale is different. We will need to adjust the values accordingly. To achieve this, we will calculate the necessary transformation matrices and apply them to the data points.

This can be done by determining the homogenous transformation matrices for each body coordinate system and transforming the values from Young's coordinate system to Johnson model's coordinate system. The transformation matrices will be derived based on the anatomical landmarks defined in both coordinate systems, ensuring accurate mapping of the data points. 

One method to achieve this is to use mesh registration.

We utilize the `open3d` library to facilitate the registration process, allowing us to align the meshes accurately based on the defined anatomical landmarks. Additionally, we implement a function to compute the transformation matrices based on the identified attachment points, ensuring that the registration process is robust and reliable.

In [6]:
from src.registration import register_meshes

# Define the source, target, and output files
mesh_dir = 'meshes'
source_postfix = '_young.stl'
target_postfix = '_johnson.stl'
output_postfix = '_y2j.stl'
output_dir = 'models/Geometry'

bodies = ['spine', 'pelvis_r', 'femur_r', 'tibia_r', 'foot_r']
transforms = {}

for body in bodies:
    source_file = f"{mesh_dir}/{body}{source_postfix}"
    target_file = f"{mesh_dir}/{body}{target_postfix}"
    output_file = f"{output_dir}/{body}{output_postfix}"

    # Register the meshes
    transform = register_meshes(source_file, target_file, output_file)
    transforms[body] = transform

Loading meshes...
Source center: [-0.02684636  0.00015972 -0.01918692], scale: 0.18391392845668425
Target center: [ 1.63168984e-02  1.05028114e-03 -8.92703315e-05], scale: 0.020141159204447458
Automatically determined parameters:
  - Normal estimation radius: 0.0500
  - Feature radius: 0.1500
  - Distance threshold: 0.0500
Computing features...
Performing global registration...
Refining registration...
Registered mesh saved to models/Geometry/spine_y2j.stl
Loading meshes...
Source center: [ 0.02837774 -0.07522624  0.01979352], scale: 0.24684772746176487
Target center: [ 0.00265115 -0.00026925 -0.00322137], scale: 0.025033484688431206
Automatically determined parameters:
  - Normal estimation radius: 0.0418
  - Feature radius: 0.1044
  - Distance threshold: 0.0500
Computing features...
Performing global registration...
Refining registration...
Registered mesh saved to models/Geometry/pelvis_r_y2j.stl
Loading meshes...
Source center: [-0.06812427 -0.0240565   0.15362877], scale: 0.197257

Now we update the opensim model with the new meshes. You could do this programmatically, but I didn't want to deal with the OpenSim API for this.

In [ ]:
from src.registration import convert_points_between_meshes
import numpy as np
import pandas as pd
import opensim as osim

data_dir = 'data'

source_file = f"{data_dir}/young_model_attachments.csv"
# Load the source attachment points -- Note: MUST BE IN ORDER FROM ORIGIN TO INSERTION
source_df = pd.read_csv(source_file)
target_file = f"{data_dir}/y2j_attachments.csv"

model_file = 'models/rat_hindlimb_millard_young.osim'
model = osim.Model(model_file)
muscles : osim.SetMuscles = model.getMuscles()

for i in range(muscles.getSize()):
    muscle : osim.Muscle = muscles.get(i)
    muscle_name = muscle.getName()
    
    geo_path : osim.GeometryPath = muscle.getGeometryPath()
    path_points : osim.PathPointSet = geo_path.getPathPointSet()    
    existing_path_points = path_points.getSize()
    
    path_wraps = geo_path.getWrapSet()
    existing_wraps = path_wraps.getSize()
    # Remove the old path wraps
    for j in range(existing_wraps - 1, -1, -1):
        path_wraps.remove(j)
        print(f"Removed path wrap index: {j} for muscle: {muscle_name}")
    
    # Add the new path points
    muscle_rows = source_df[source_df['Abbreviation'] == muscle_name]
    index = 1
    for _, row in muscle_rows.iterrows():
        lower_frame = row['Frame'].lower()
        frame = lower_frame + '_r' if lower_frame != 'spine' else lower_frame
        body = model.getBodySet().get(frame)
        point_name = muscle_name + '_' + frame + '_' + row['Type'].lower() + '_' + str(index)
        young_loc = np.array([[row['X (mm)'], row['Y (mm)'], row['Z (mm)']]])/100 # Young has a weird scale
        loc = convert_points_between_meshes(young_loc, transforms[frame])
        vec = osim.Vec3(loc[0][0], loc[0][1], loc[0][2])
        muscle.addNewPathPoint(point_name, body, vec)
        print(f"Added path point for muscle: {muscle_name}, body: {body.getName()}, location: {loc}")
        index += 1
    # Remove the old path points
    for j in range(existing_path_points - 1, -1, -1):
        path_points.remove(j)
        print(f"Removed path point index: {j} for muscle: {muscle_name}")

# Make sure to update the model before finalizing connections
model.finalizeFromProperties()
model.finalizeConnections()

# Save the modified model
model.printToXML('models/rat_hindlimb_millard_y2j.osim')

[info] Loaded model RatHindlimbRight from file models/rat_hindlimb_millard_young.osim
Removed path wrap index: 1 for muscle: BFa
Removed path wrap index: 0 for muscle: BFa
Added path point for muscle: BFa, body: pelvis_r, location: [[-0.01343989  0.00981751 -0.00651766]]
Added path point for muscle: BFa, body: pelvis_r, location: [[-0.01361999  0.00542459 -0.00184753]]
Added path point for muscle: BFa, body: pelvis_r, location: [[-0.01368217  0.0013663  -0.00027909]]
Added path point for muscle: BFa, body: tibia_r, location: [[-0.00348673  0.03196627  0.00149422]]
Removed path point index: 1 for muscle: BFa
Removed path point index: 0 for muscle: BFa
Removed path wrap index: 1 for muscle: BFp
Removed path wrap index: 0 for muscle: BFp
Added path point for muscle: BFp, body: pelvis_r, location: [[-0.01641398  0.00379038 -0.00237626]]
Added path point for muscle: BFp, body: tibia_r, location: [[-0.00366277  0.02624208  0.00056152]]
Added path point for muscle: BFp, body: tibia_r, locatio

True

## Final Step: Create Bilateral Model

In [ ]:
# from src import mirror_hindlimb
# #TODO: Implement the mirror_hindlimb function to create a bilateral model
# bilateral_model : osim.Model = mirror_hindlimb(model)

# # Save the mirrored model
# bilateral_model.printToXML('rat_hindlimb_bilateral.osim')